# Connect Your Agent to Bioinformatics Tools

This notebook auto-detects how your agent framework registers and executes tools,
then generates wrappers + adapter to inject our bioinformatics tools (bqtools, kaptain, bioemu, etc.).

**Steps:**
1. Enter your agent's GitHub URL (or upload a zip)
2. Auto-scan the agent source to find its tool registration interface
3. Generate wrappers and adapter
4. Inject tools and test with a real bioinformatics task

In [ ]:
#@title Step 1: Configure your agent { display-mode: "form" }
#@markdown Enter your agent's GitHub repo URL (or leave empty to upload a zip)
AGENT_REPO_URL = "" #@param {type:"string"}
#@markdown Or upload a zip file of your agent code (skip if you filled in the URL above)
#@markdown ---

import os, subprocess, sys

TARGET_DIR = "/content/my_agent"

if AGENT_REPO_URL:
    print(f"Cloning {AGENT_REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", AGENT_REPO_URL, TARGET_DIR],
                   check=True, capture_output=True, text=True)
elif os.path.exists(TARGET_DIR):
    print(f"Using existing agent at {TARGET_DIR}")
else:
    print("No URL provided. Please upload your agent as a zip file:")
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            os.makedirs("/content/_agent_upload", exist_ok=True)
            for fn in uploaded.keys():
                if fn.endswith(".zip"):
                    subprocess.run(["unzip", "-o", "-q", fn, "-d", "/content/_agent_upload"],
                                   check=True)
            subs = [d for d in os.listdir("/content/_agent_upload")
                    if os.path.isdir(os.path.join("/content/_agent_upload", d))]
            TARGET_DIR = os.path.join("/content/_agent_upload", subs[0]) if len(subs) == 1 else "/content/_agent_upload"
    except Exception as e:
        print("Upload skipped:", e)

if not os.path.exists(TARGET_DIR):
    # Default: clone Biomni as demo
    print("No agent provided → using Biomni as demo")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/snap-stanford/Biomni.git", TARGET_DIR],
                   check=True, capture_output=True, text=True)

print(f"Agent directory: {TARGET_DIR}")
assert os.path.exists(TARGET_DIR), "Agent directory not found"

In [ ]:
#@title Step 2: Install dependencies { display-mode: "form" }
import os, subprocess, sys

# Clone our tools repo if needed
TOOLS_DIR = "/content/scientific_training2_cxy"
if not os.path.isdir(os.path.join(TOOLS_DIR, ".git")):
    print("Cloning tool registry repo...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/caixiaoyao2025/scientific_training2_cxy.git",
                    TOOLS_DIR], check=True, capture_output=True, text=True)
else:
    subprocess.run(["git", "-C", TOOLS_DIR, "pull", "--ff-only"],
                   capture_output=True, text=True)

sys.path.insert(0, TOOLS_DIR)
os.chdir(TOOLS_DIR)

# Install agent_connector deps
!pip install -q pyyaml 2>/dev/null

print("Tools repo ready:", TOOLS_DIR)
import yaml
reg_path = os.path.join(TOOLS_DIR, "data", "mcp_registry.yaml")
if os.path.exists(reg_path):
    tools = yaml.safe_load(open(reg_path, encoding="utf-8"))["tools"]
    print(f"Available tools: {[t['name'] for t in tools]}")
else:
    print("WARNING: data/mcp_registry.yaml not found")

In [ ]:
#@title Step 3: Scan agent & generate wrappers { display-mode: "form" }
import sys, os, json
sys.path.insert(0, "/content/scientific_training2_cxy")

from agent_connector.scanner import build_schema
from agent_connector.generator import generate_wiring, load_wrappers, load_adapter
import yaml

# Load tools from registry
reg_path = os.path.join(TOOLS_DIR, "data", "mcp_registry.yaml")
tools = yaml.safe_load(open(reg_path, encoding="utf-8"))["tools"]

# Scan the agent
print("Scanning agent source code...")
schema = build_schema(TARGET_DIR, include_evidence=False)

print(f"\nDetected agent class:     {schema.get('agent_class')}")
print(f"Registration method:      {schema.get('registration_method')}")
print(f"Registration style:       {schema.get('registration_style')}")
print(f"Execution method:         {schema.get('execution_method')}")
print(f"Wiring style:             {schema.get('wiring_style')}")
print(f"Confidence:               {schema.get('confidence')}")

# Generate wiring
OUT_DIR = os.path.join(TOOLS_DIR, "wiring")
wiring = generate_wiring(tools, schema, out_dir=OUT_DIR)
print(f"\nWiring mode: {wiring['mode']}")
print(f"Artifacts:   {list(wiring['artifacts'].keys())}")
print(f"\nInstructions: {wiring['instructions']}")

In [ ]:
#@title Step 4: Inject tools into your agent { display-mode: "form" }
import sys, os, importlib
sys.path.insert(0, "/content/scientific_training2_cxy")

# Load wrappers
out_dir = os.path.join(TOOLS_DIR, "wiring")
pkg_dir = os.path.join(out_dir, "generated_tools")
if pkg_dir not in sys.path:
    sys.path.insert(0, pkg_dir)

registration_style = schema.get("registration_style") or "object"
wrappers = load_wrappers(package_name="generated_tools",
                         registration_style=registration_style)
print(f"Loaded {len(wrappers)} tool wrappers")
for w in wrappers[:5]:
    name = getattr(w, "name", None) or getattr(w, "__name__", str(w))
    print(f"  - {name}")
if len(wrappers) > 5:
    print(f"  ... and {len(wrappers) - 5} more")

# Try to instantiate agent and inject tools
agent = None
reg_method = schema.get("registration_method")

if reg_method and wiring["artifacts"].get("adapter"):
    try:
        # Find and import the agent class
        cls_name = schema.get("agent_class")
        # Search for the class in the agent repo
        for dirpath, _, filenames in os.walk(TARGET_DIR):
            for fn in filenames:
                if not fn.endswith(".py"):
                    continue
                p = os.path.join(dirpath, fn)
                try:
                    import ast
                    tree = ast.parse(open(p, encoding="utf-8-sig").read().lstrip("\ufeff"))
                except Exception:
                    continue
                for node in tree.body:
                    if isinstance(node, ast.ClassDef) and node.name == cls_name:
                        rel = os.path.relpath(p, TARGET_DIR)
                        mod_name = os.path.splitext(rel)[0].replace(os.sep, ".")
                        sys.path.insert(0, TARGET_DIR)
                        cls = getattr(importlib.import_module(mod_name), cls_name)
                        try:
                            agent = cls()
                        except TypeError:
                            agent = None
                        break
            if agent:
                break

        if agent:
            adapter_path = wiring["artifacts"]["adapter"]
            Adapter = load_adapter(cls_name, adapter_path=os.path.abspath(adapter_path))
            Adapter(agent).install_tools(wrappers)
            print(f"\nInjected {len(wrappers)} tools into {type(agent).__name__}")
        else:
            print(f"\nCould not instantiate {cls_name}")
    except Exception as e:
        print(f"\nInjection failed: {type(e).__name__}: {e}")

# Fallback: DynamicAgent
if agent is None:
    print("\nFalling back to DynamicAgent (wrapper list only)")
    class DynamicAgent:
        def __init__(self): self.tools = []
        def add_tool(self, t): self.tools.append(t)
    agent = DynamicAgent()
    for w in wrappers:
        agent.add_tool(w)
    print(f"DynamicAgent has {len(agent.tools)} tools")

print(f"Agent type: {type(agent).__name__}")

In [ ]:
#@title Step 5: Test with a bioinformatics task { display-mode: "form" }
#@markdown Run a sample tool to verify the integration works
import sys, os
sys.path.insert(0, "/content/scientific_training2_cxy")

# Create a sample FASTA file
sample_fasta = "/content/test_sample.fasta"
with open(sample_fasta, ",".encode() if False else "w", encoding="utf-8") as f:
    f.write(">seq1\nACGT\nACGT\n>seq2\nTTTTTT\n>seq3\nCCCGGG\n>seq4\nAAAAT\n>seq5\nGATAC\n")
print(f"Sample FASTA: {sample_fasta}")

# Find and invoke a tool
wmap = {}
for w in wrappers:
    nm = getattr(w, "name", None) or getattr(w, "__name__", None)
    if nm:
        wmap[nm] = w

print(f"\nAvailable wrapped tools: {list(wmap.keys())[:10]}")

# Try fasta_stats or any tool with a fasta_path param
test_tool = wmap.get("fasta_contig_stats_python") or wmap.get("fasta_stats")
if test_tool:
    print(f"\nRunning {getattr(test_tool, 'name', getattr(test_tool, '__name__', '?'))}...")
    try:
        if hasattr(test_tool, "run"):
            result = test_tool.run(fasta_path=sample_fasta)
        elif callable(test_tool):
            result = test_tool(fasta_path=sample_fasta)
        else:
            result = str(test_tool)
        print(f"Result: {str(result)[:500]}")
    except Exception as e:
        print(f"Error: {type(e).__name__}: {e}")
else:
    # Just call the first available tool
    if wrappers:
        w = wrappers[0]
        name = getattr(w, "name", getattr(w, "__name__", "?"))
        print(f"\nNo fasta_stats found, testing first tool: {name}")
        try:
            if hasattr(w, "run"):
                result = w.run(fasta_path=sample_fasta)
            else:
                result = w(fasta_path=sample_fasta)
            print(f"Result: {str(result)[:500]}")
        except Exception as e:
            print(f"Error: {type(e).__name__}: {e}")
    else:
        print("No wrappers available")

## What happened?

1. **Scanner** analyzed your agent's source code and found:
   - The agent class name
   - How it registers tools (e.g., `add_tool`, `register`, `bind_tools`)
   - How it executes tools (e.g., `run`, `invoke`, `execute`)
   - The wiring style (adapter, manifest, config, or prompt)

2. **Generator** produced:
   - **Wrappers**: one per tool, matching your agent's registration style
   - **Adapter**: calls your agent's registration method for each wrapper
   - Or a **manifest/config/prompt block** if no register method was found

3. **Injection** loaded the adapter and called `agent.add_tool(wrapper)` for each tool.

### Supported wiring styles

| Style | When | What's generated |
|-------|------|------------------|
| **adapter** | Agent has `add_tool()` / `register()` method | Class/function wrappers + adapter.py |
| **manifest** | Agent uses `tools=[...]` in LLM calls | tools_manifest.json (OpenAI function calling format) |
| **config** | Agent reads tools from config file | tools_config.yaml |
| **prompt** | Agent uses system prompt for tools | Text block to append to system prompt |